# Fine-Tuning GLiNER for CTI NER — Colab Runner

NLP capstone companion notebook for the `scrape_cisa` project. Trains and evaluates
`urchade/gliner_medium-v2.1` on the CISA advisory silver corpus.

**Inputs (upload when prompted, or place in `/content/`):**
- `gliner_train.jsonl` — training chunks with `slug`/`chunk_start` (built by `scraper/create_training_data.py`)
- `split_slugs.json` — document-level split manifest (built by `scraper/finetune_gliner.py`)

Using the committed manifest keeps Colab results comparable to local runs — do not resplit here.

**Runtime guidance:**
- **GPU (recommended):** any CUDA runtime works; fp16 enabled automatically. A T4 trains 5 epochs in minutes.
- **TPU (experimental):** PyTorch reaches TPU via `torch_xla`. GLiNER pads batches dynamically, so XLA recompiles per unique shape — first epoch may be pathologically slow. Provided for work-account TPU experiments; measure before committing to it.
- **CPU:** works (this is how the local reference run trains), just slow.

Code cells marked `SYNC` mirror `scraper/evaluate.py` / `scraper/finetune_gliner.py` in the repo — if you change logic here, port it back.

In [ ]:
%pip -q install gliner==0.2.26
# TPU runtime only (skip on GPU/CPU): torch_xla ships preinstalled on Colab TPU
# runtimes; if missing, uncomment:
# %pip -q install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

In [ ]:
import torch

DEVICE_KIND = "cpu"
try:
    import torch_xla.core.xla_model as xm  # noqa: F401
    DEVICE_KIND = "tpu"
except ImportError:
    if torch.cuda.is_available():
        DEVICE_KIND = "cuda"

USE_FP16 = DEVICE_KIND == "cuda"
print(f"Runtime: {DEVICE_KIND} | fp16: {USE_FP16}")
if DEVICE_KIND == "tpu":
    print("WARNING: dynamic batch shapes force per-shape XLA recompilation; "
          "expect a very slow first epoch. GPU runtime is the pragmatic choice.")

In [ ]:
# Fetch inputs if not already present (Colab upload dialog)
import os

NEEDED = ["gliner_train.jsonl", "split_slugs.json"]
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    from google.colab import files  # type: ignore
    print(f"Upload: {missing}")
    files.upload()
assert all(os.path.exists(f) for f in NEEDED), "data files missing"

In [ ]:
# SYNC: data loading + document-level split (scraper/finetune_gliner.py)
import json

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

records = load_jsonl("gliner_train.jsonl")
split = json.load(open("split_slugs.json"))
member = {s: name for name, ss in split.items() for s in ss}

buckets = {"train": [], "val": [], "test": []}
for rec in records:
    buckets[member[rec["slug"]]].append(rec)

def to_trainer_format(recs):
    return [{"tokenized_text": r["tokenized_text"], "ner": r["ner"]} for r in recs]

LABELS = sorted({l for rec in records for _, _, l in rec["ner"]})
print({k: f"{len({r['slug'] for r in v})} docs / {len(v)} chunks" for k, v in buckets.items()})
print(f"{len(LABELS)} labels: {LABELS}")

In [ ]:
# SYNC: span-level evaluation (scraper/evaluate.py)
from collections import defaultdict

def token_char_offsets(tokens):
    offsets, pos = [], 0
    for tok in tokens:
        offsets.append((pos, pos + len(tok)))
        pos += len(tok) + 1
    return offsets

def char_span_to_token_span(cs, ce, offsets):
    first = last = None
    for i, (ts, te) in enumerate(offsets):
        if ts < ce and te > cs:
            if first is None:
                first = i
            last = i
    return None if first is None else (first, last)

def gold_document_spans(doc_records):
    return {(r["chunk_start"] + s, r["chunk_start"] + e, l)
            for r in doc_records for s, e, l in r["ner"]}

def predict_document_spans(model, doc_records, labels, threshold=0.4):
    spans = set()
    for rec in doc_records:
        tokens, base = rec["tokenized_text"], rec["chunk_start"]
        offsets = token_char_offsets(tokens)
        for p in model.predict_entities(" ".join(tokens), labels, threshold=threshold):
            ts = char_span_to_token_span(p["start"], p["end"], offsets)
            if ts:
                spans.add((base + ts[0], base + ts[1], p["label"]))
    return spans

def match_relaxed(gold, pred):
    matched = 0
    by_g, by_p = defaultdict(list), defaultdict(list)
    for g in gold: by_g[g[2]].append(g)
    for p in pred: by_p[p[2]].append(p)
    for label, gspans in by_g.items():
        pspans = sorted(by_p.get(label, []))
        used = [False] * len(pspans)
        for gs, ge, _ in sorted(gspans):
            for j, (ps, pe, _) in enumerate(pspans):
                if not used[j] and ps <= ge and pe >= gs:
                    used[j] = True; matched += 1; break
    return matched

class Scorer:
    def __init__(self):
        self.tp, self.fp, self.fn = defaultdict(int), defaultdict(int), defaultdict(int)
    def add_strict(self, gold, pred):
        for g in gold:
            (self.tp if g in pred else self.fn)[g[2]] += 1
        for p in pred:
            if p not in gold: self.fp[p[2]] += 1
    def add_relaxed(self, gold, pred):
        for label in {s[2] for s in gold | pred}:
            g = {s for s in gold if s[2] == label}
            p = {s for s in pred if s[2] == label}
            m = match_relaxed(g, p)
            self.tp[label] += m; self.fn[label] += len(g) - m; self.fp[label] += len(p) - m
    def report(self):
        per = {}
        for label in sorted(set(self.tp) | set(self.fp) | set(self.fn)):
            tp, fp, fn = self.tp[label], self.fp[label], self.fn[label]
            p = tp / (tp + fp) if tp + fp else 0.0
            r = tp / (tp + fn) if tp + fn else 0.0
            f = 2 * p * r / (p + r) if p + r else 0.0
            per[label] = {"P": p, "R": r, "F1": f, "support": tp + fn}
        tp, fp, fn = sum(self.tp.values()), sum(self.fp.values()), sum(self.fn.values())
        mp = tp / (tp + fp) if tp + fp else 0.0
        mr = tp / (tp + fn) if tp + fn else 0.0
        return {"per_label": per,
                "micro": {"P": mp, "R": mr,
                          "F1": 2 * mp * mr / (mp + mr) if mp + mr else 0.0},
                "macro_F1": sum(v["F1"] for v in per.values()) / len(per) if per else 0.0}

def evaluate(model, subset="test", threshold=0.4):
    by_doc = defaultdict(list)
    for rec in buckets[subset]:
        by_doc[rec["slug"]].append(rec)
    strict, relaxed = Scorer(), Scorer()
    for slug, docs in sorted(by_doc.items()):
        gold = gold_document_spans(docs)
        pred = predict_document_spans(model, docs, LABELS, threshold)
        strict.add_strict(gold, pred)
        relaxed.add_relaxed(gold, pred)
        print(f"  {slug}: {len(gold)} gold / {len(pred)} pred")
    return strict.report(), relaxed.report()

def print_table(name, rep):
    print(f"\n{name}")
    print(f"  {'label':<30} {'P':>6} {'R':>6} {'F1':>6} {'support':>8}")
    for label, m in sorted(rep["per_label"].items(), key=lambda x: -x[1]["support"]):
        print(f"  {label:<30} {m['P']:>6.2f} {m['R']:>6.2f} {m['F1']:>6.2f} {m['support']:>8}")
    mi = rep["micro"]
    print(f"  {'MICRO':<30} {mi['P']:>6.2f} {mi['R']:>6.2f} {mi['F1']:>6.2f}")
    print(f"  {'MACRO F1':<30} {'':>6} {'':>6} {rep['macro_F1']:>6.2f}")

In [ ]:
# Zero-shot baseline
from gliner import GLiNER

BASE = "urchade/gliner_medium-v2.1"
base_model = GLiNER.from_pretrained(BASE)
if DEVICE_KIND == "cuda":
    base_model = base_model.to("cuda")

zs_strict, zs_relaxed = evaluate(base_model)
print_table("ZERO-SHOT STRICT", zs_strict)
print_table("ZERO-SHOT RELAXED", zs_relaxed)

In [ ]:
# SYNC: fine-tune (scraper/finetune_gliner.py). Same hyperparameters as the
# local reference run: 5 epochs, LR 5e-5 (encoder 1e-5), effective batch 32.
from gliner.data_processing.collator import SpanDataCollator
from gliner.training import Trainer, TrainingArguments

model = GLiNER.from_pretrained(BASE)
collator = SpanDataCollator(model.config, data_processor=model.data_processor,
                            prepare_labels=True)

args = TrainingArguments(
    output_dir="gliner-cti-v1",
    learning_rate=5e-5, weight_decay=0.01,
    others_lr=1e-5, others_weight_decay=0.01,
    lr_scheduler_type="linear", warmup_ratio=0.1,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, num_train_epochs=5,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, save_total_limit=2, logging_steps=50,
    seed=42, fp16=USE_FP16, dataloader_num_workers=0,
    report_to="none", remove_unused_columns=False,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=to_trainer_format(buckets["train"]),
                  eval_dataset=to_trainer_format(buckets["val"]),
                  data_collator=collator)
trainer.train()
model.save_pretrained("gliner-cti-v1")

In [ ]:
# Fine-tuned evaluation + comparison
ft_model = GLiNER.from_pretrained("gliner-cti-v1")
if DEVICE_KIND == "cuda":
    ft_model = ft_model.to("cuda")

ft_strict, ft_relaxed = evaluate(ft_model)
print_table("FINE-TUNED STRICT", ft_strict)
print_table("FINE-TUNED RELAXED", ft_relaxed)

print(f"\n{'label':<30} {'ZS F1':>7} {'FT F1':>7} {'delta':>7}")
for label in sorted(set(zs_strict["per_label"]) | set(ft_strict["per_label"])):
    z = zs_strict["per_label"].get(label, {}).get("F1", 0.0)
    f = ft_strict["per_label"].get(label, {}).get("F1", 0.0)
    print(f"{label:<30} {z:>7.2f} {f:>7.2f} {f - z:>+7.2f}")
print(f"{'MICRO':<30} {zs_strict['micro']['F1']:>7.2f} "
      f"{ft_strict['micro']['F1']:>7.2f} "
      f"{ft_strict['micro']['F1'] - zs_strict['micro']['F1']:>+7.2f}")

In [ ]:
# Export the checkpoint (zip download; or copy to Drive)
import shutil
shutil.make_archive("gliner-cti-v1", "zip", "gliner-cti-v1")
print("gliner-cti-v1.zip ready — download via the Files panel, or:")
print("  from google.colab import files; files.download('gliner-cti-v1.zip')")